# ADNI maximum-slot parallel replication — 3 folds × 3 seeds, no augmentation

**504 total runs, preserving completed and interrupted work:** four models × 14 conditions × folds 0/1/2 × seeds 0/1/2.
Each fold retains its saved 5% subject subset: 5 training subjects/10 crops and 27 validation subjects/54 crops.
Left/right hippocampi stay together; subjects never overlap between training and validation within a fold.
The cluster pilot remains provenance. Completed Colab runs in the existing results folder are reused.

- **No augmentation:** no random flips, rotations, intensity jitter, or test-time augmentation.
  The existing deterministic normalization/padding stays unchanged.
- nnUNet: native foreground Dice+CE, original deep supervision and SGD; 60% auxiliary targets, 75% p95 ceiling.
- SwinUNETR, UNETR and 3D U-Net: established Dice including background and AdamW; 10% targets, 50% p95 ceiling.
- Preserve BF16 precision, batch size 1, initialization, LR schedule, calibration, SDF timing and the
  31,976,450,667,090 optimization-FLOP cap per run. Calibration and preflight compute are separate.

Keep your existing `adni_data_verified.zip`, `adni_replication_code.zip` and results on Drive. No dataset re-upload is needed. Use the same **A100-SXM4-80GB** and software environment that passed your validation; the evidence gate checks the exact versions.

Run cells in order. All results/recovery go directly to Drive; your Mac is not an archive server.
Colab availability, session lifetime and background execution depend on your plan. Closing a laptop
is not a guarantee of uninterrupted Colab execution. See the [official FAQ](https://research.google.com/colaboratory/faq.html).


**Switching from the existing notebook:** interrupt its run cell once and wait until checkpoint
uploads finish. Keep the same A100 type, environment and results directory. Upload only
`adni_validated_acceleration_control_v5.zip` beside the two original ZIPs; open this notebook and run its cells in order.
The original science bundle is unchanged. Your returned A100 equivalence checks passed for all four
models. This package enables exactly that tested acceleration code. Verified results are skipped, recovery resumes first,
and the existing preflight is required and reused, with a separate read-only acceleration evidence gate. Reuse the current runtime when possible. If Colab assigns a
fresh A100 runtime, verified recovery is restored from Drive. Never delete the old runtime before
its clean stop and recovery uploads finish; do not leave two sweeps running against these results.
**Do not switch while the old run is active.** This package does not stop it for you.
No dataset upload is needed; leave the existing data/code ZIPs and results exactly where they are.
Setup verifies an existing local bundle in place; on a fresh VM it stages the same existing ZIPs.
Local process locks and a persistent Drive ownership marker reject visible competing controllers.
After an unclean VM loss, a leftover ownership marker needs inspection; it is never expired or
deleted automatically. Drive mounts are not a distributed lock service: use one Colab runtime
for this results folder, and never launch controllers simultaneously from separate VMs.

`MAX_WORKERS=0` targets the CPU-supported maximum immediately: two threads per worker and
two CPU cores reserved for orchestration. Slots fill as soon as memory, storage and baseline
dependencies permit, with no solo memory runs or throughput ramp. The target does not fall
when GPU utilization or measured throughput changes. Actual occupancy can still be lower.
New checkpoints and recovery staging use a separately mounted `/mnt/local-scratch` when
present; existing run directories/checkpoints remain where they are. All durable charges and
recovery uploads remain unchanged. Exact GPU equivalence is verified on your A100; total sweep
speedup and maximum-worker throughput have not yet been measured.


## 1. Mount Drive and stage the unchanged science bundle plus scheduler


In [ ]:
from contextlib import contextmanager
import json, threading, time
from pathlib import Path
@contextmanager
def visible_phase(label):
    print('PHASE:', label, flush=True)
    stop = threading.Event()
    start = time.monotonic()
    path = Path('/content/hippo_adni_setup_status.json')
    def heartbeat():
        while not stop.wait(30):
            value = dict(phase=label, elapsed_seconds=time.monotonic()-start,
                         status='still waiting; this heartbeat is not evidence of progress')
            path.write_text(json.dumps(value))
            print('HEARTBEAT:', label, 'still waiting', round(value['elapsed_seconds']), 'seconds', flush=True)
    path.write_text(json.dumps(dict(phase=label, status='started')))
    thread = threading.Thread(target=heartbeat, daemon=True); thread.start()
    try:
        yield
    finally:
        stop.set(); thread.join()
        path.write_text(json.dumps(dict(phase=label, status='phase ended', elapsed_seconds=time.monotonic()-start)))
from google.colab import drive
with visible_phase('mount Google Drive'):
    drive.mount('/content/drive')
from pathlib import Path
import fcntl, hashlib, json, os, shutil, signal, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/hippo_adni_3fold_3seed')
INPUT = DRIVE_ROOT/'input'
RESULTS = DRIVE_ROOT/'results_noaug_v1'  # Preserve this exact existing results folder.
BUNDLE = Path('/content/hippo_adni_bundle')
CONTROL = Path('/content/hippo_adni_adaptive')
SCIENTIFIC_SHA = 'e0b87c00231bb355aa66531f6684c361a632b3f8e393f6afbfb4939a37d0caeb'
EXPECTED = {'adni_data_verified.zip':'e05bb92b0a6f5024dc9c823283a63020dba8f9cbef70141949a854f6bad6e709', 'adni_replication_code.zip':'1a1c1361f371e4b2f76f878f298aa4ca884627a4352c399e4b854daa79bfb436'}
CONTROL_SHA = '850ac7bd1ea61325f7226f8c38f4a238bd6f79438afb67e661ff0bfd76335b6c'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''):
            h.update(block)
    return h.hexdigest()

BUNDLE.mkdir(exist_ok=True)
(BUNDLE/'work').mkdir(exist_ok=True)
with visible_phase('open existing Drive results folder'):
    RESULTS.mkdir(parents=True, exist_ok=True)
def unpack_verified(src, expected, destination):
    local = Path('/content')/src.name
    with visible_phase('stage and hash-check ' + src.name):
        shutil.copyfile(src, local)
        assert sha256(local) == expected, f'Upload hash mismatch: {src.name}'
    with zipfile.ZipFile(local) as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            assert not member.is_absolute() and '..' not in member.parts
            assert not ((info.external_attr >> 16) & 0o170000) == 0o120000
        archive.extractall(destination)

with (BUNDLE/'work/sweep.lock').open('a') as lock:
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError('The old sweep is running. Interrupt it once and wait for recovery before switching.')
    for proc in Path('/proc').glob('[0-9]*/cmdline'):
        try:
            words = proc.read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            continue
        if str(BUNDLE/'entry.py') in words or (str(BUNDLE) in words and any(w.endswith('/worker.py') for w in words)):
            raise RuntimeError('An existing trainer is still running. Wait for its clean stop before setup.')
    manifest = BUNDLE/'bundle_manifest.json'
    print('PHASE: verify existing local bundle or stage original uploads', flush=True)
    if manifest.exists():
        assert sha256(manifest) == SCIENTIFIC_SHA, 'Existing bundle differs; stop and preserve it for inspection.'
        with visible_phase('hash-check frozen local science files'):
            for name, expected in json.loads(manifest.read_text()).items():
                assert sha256(BUNDLE/name) == expected, f'Existing bundle file differs: {name}'
        print('Verified existing scientific bundle in place; work/checkpoints preserved.')
    else:
        assert not any((BUNDLE/'work').glob('*/budget_ledger.json')), 'Existing work without a manifest needs inspection.'
        for name, expected in EXPECTED.items():
            unpack_verified(INPUT/name, expected, BUNDLE)
        assert sha256(manifest) == SCIENTIFIC_SHA
        print('Staged original bundle; existing Drive results/recovery preserved.')
    CONTROL.mkdir(exist_ok=True)
    unpack_verified(INPUT/'adni_validated_acceleration_control_v5.zip', CONTROL_SHA, CONTROL)
print('No augmentation. All input data stays on local SSD; results stay in', RESULTS)


## 2. Check Drive headroom
Reserve at least 8 GiB for current recovery snapshots and metadata. Completed model checkpoints
are retired only after their evaluated results have been archived and verified.

Concurrency reserves 8 GiB of Drive headroom per active worker and 12 GiB of local SSD per worker.


In [ ]:
with visible_phase('authenticate and check Drive quota'):
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    creds, _ = google.auth.default()
    quota = build('drive', 'v3', credentials=creds, cache_discovery=False).about().get(fields='storageQuota').execute()['storageQuota']
    if 'limit' in quota:
        free = int(quota['limit']) - int(quota['usage'])
        print(f'Drive quota headroom: {free/1024**3:.2f} GiB')
        assert free >= 8*1024**3, 'Free at least 8 GiB before training.'
    else:
        print('Drive reports no fixed quota limit.')
    assert shutil.disk_usage('/content').free >= 20*1024**3, 'At least 20 GiB runtime SSD headroom required.'


## 3. Create an isolated training environment
The notebook kernel is left intact. Package pins match the cluster recipe, including actual
dynamic-network-architectures 0.4.4. Installation is reused while this runtime survives.
Missing pip is bootstrapped from PyPA. Setup and campaign output is printed and saved locally.
CUDA initialization, binary losses and checkpoint resume are checked in the next cell.


In [ ]:
VENV = Path('/content/hippo_adni_venv')
PYTHON = VENV/'bin/python'
def run_setup_logged(args, log_path):
    log_path = Path(log_path)
    tail = []
    from contextlib import nullcontext
    context = nullcontext() if str(CONTROL/'scheduler.py') in args else visible_phase('subprocess setup or verification')
    with context, log_path.open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1, start_new_session=True)
        def drain():
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
                tail.append(line)
                del tail[:-40]
        interrupted = False
        try:
            drain()
            result = proc.wait()
        except KeyboardInterrupt:
            interrupted = True
            print('Requesting a clean stop. Wait for the process and recovery upload to finish.')
            os.killpg(proc.pid, signal.SIGINT)
            drain()
            result = proc.wait()
        finally:
            proc.stdout.close()
        if result:
            raise RuntimeError(f'Command exited {result}. Full log: {log_path}\n' + ''.join(tail))

def ensure_training_pip(python, bootstrap, log_path):
    probe = subprocess.run([str(python), '-m', 'pip', '--version'], capture_output=True, text=True)
    if probe.returncode:
        import urllib.request
        print('PHASE: bootstrap pip from PyPA', flush=True)
        urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', str(bootstrap))
        run_setup_logged([str(python), str(bootstrap)], log_path)

if not PYTHON.exists():
    run_setup_logged([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)], '/content/hippo_adni_setup.log')
ensure_training_pip(PYTHON, Path('/content/hippo_adni_get_pip.py'), '/content/hippo_adni_setup.log')
run_setup_logged([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(BUNDLE/'requirements-colab.txt')], '/content/hippo_adni_setup.log')
def campaign(command, *extra):
    args = ([str(PYTHON), '-u', str(CONTROL/'scheduler.py'), command, '--bundle', str(BUNDLE), '--durable', str(RESULTS), *map(str, extra)]
            if command in ('run', 'report') else [str(PYTHON), '-u', str(BUNDLE/'runtime.py'), command, '--bundle', str(BUNDLE), '--durable', str(RESULTS), *map(str, extra)])
    if command == 'preflight':
        args = [str(PYTHON), '-u', str(CONTROL/'acceleration_gate.py'), '--bundle', str(BUNDLE), '--durable', str(RESULTS)]
    if command == 'report':
        args += ['--max-runs', '0']
    run_setup_logged(args, Path('/content')/f'hippo_adni_{command}.log')
campaign('verify')
subprocess.run(['nvidia-smi'], check=True)


## 4. Reuse the existing preflight and verify your A100 acceleration evidence

This cell reads the original passed preflight and environment binding, verifies the unchanged
scientific bundle, and checks the exact original/candidate equivalence evidence you returned
for all four models. It does not rerun GPU diagnostics or add training/benchmarks.
The binary ADNI loss graphs, saved subject splits and all scientific settings remain unchanged.
A GPU/software mismatch blocks execution without changing receipts or accounting.


In [ ]:
campaign('preflight')


## 5. Run or resume with maximum admissible workers

`MAX_WORKERS = 0` immediately targets `(available_CPU_cores - 2) // 2` workers, at least one.
A positive value sets a lower ceiling. GPU utilization and throughput are observations only:
no gradual ramp, mandatory solo memory run, or automatic target reduction is applied.
Eligible slots refill after each completed worker's archive is verified and cleanup is recorded.
Unknown conditions receive provisional allowances of at least 12 GiB VRAM and 8 GiB RAM,
enlarged by same-model ADNI measurements and live worker peaks. These are conservative
reservations, not validated upper bounds. Known conditions reserve twice the recorded GPU
peak plus 2 GiB, and 1.5 times the RAM peak plus 1 GiB (minimum 4 GiB each).
Every active worker's future growth is reserved, with an additional 8 GiB/15% GPU margin,
4 GiB system RAM, 8 GiB Drive space per worker, and 12 GiB per worker on each occupied disk.
Compatible, hash-verified ADNI observer records are reused across controller versions. Old
unaccelerated records gain 1 GiB of cache allowance and keep the provisional 12/8 GiB floor
until a complete accelerated scientific run supplies its own measured peaks. Diagnostic
memory peaks are never used as full-run profiles.
No other dataset's memory measurements are used. Every unfilled slot has a visible reason:
resource headroom, matched-baseline dependency, archiving/recovery, remaining tasks or stopping.

`/mnt/local-scratch` must be a real separate mount to be selected. Only missing run directories
are created there, with their original logical paths retained. Existing work is never migrated.
Recovery hardlinks and tar staging use the physical checkpoint filesystem. If the separate
disk is unavailable, the controller clearly falls back to the runtime disk. Ownership markers,
the shared local lock and the Drive controller lock remain mandatory.

Startup and the report cell list saved folders once, read each required archive file once into
a fresh disposable local snapshot, and run all unchanged hash, size, readability, identity,
source-binding, acknowledgment and cleanup checks locally. Results and memory observations
come from those verified bytes. The snapshots are removed, and a failed verification leaves
the previous report intact. Baseline proof bytes are retained only in memory for this session
and rechecked locally before the frozen initialization restoration gate.

The exact candidate you validated on the A100 is installed after the original binary recipe
is configured: preprocessed images/labels remain on GPU; immutable SDF/depth annotations
are reused; evaluation keeps identical batch size, operations and case order but defers host
metric readback; unchanged best-checkpoint writes are skipped. Latest checkpoints, durable
ledger writes/readback and recovery cadence remain intact. The frozen scientific source and
protocols are untouched; separate execution receipts record these external hooks honestly.
All four model diagnostics matched original losses, gradients, calibration, optimizer/model
state, binary metrics, RNG and checkpoint replay exactly. This evidence covers fold 0/seed 0
with all four auxiliary terms; it is not a full-sweep benchmark. No tolerances were relaxed.
The original preflight is still required. The additional evidence gate binds the exact candidate,
frozen bundle, A100-SXM4-80GB and package/CUDA/cuDNN versions from your report. A mismatch
blocks execution and preserves your data; it never resets the binding or substitutes new trials.

Existing interrupted tasks are admitted first, then fresh baselines and constraints whose own
baseline is already verified. Other eligible tasks can fill spare slots while resumed work runs.
Before each constrained run, its archived baseline initialization is verified and restored locally
for the frozen ADNI model builder. Each subject's left/right crops stay in the same saved split.
Each task still has its own original seeds, batch size 1, BF16, calibration, losses, optimizer,
update schedule, checkpoints and 31,976,450,667,090 optimization-FLOP cap. No augmentation.
Training can remain partly idle when Drive is the bottleneck; do not equate VRAM occupancy with speed.

Every 30 seconds a local observer prints the current phase, checked/verified archive counts,
completed count, active run IDs, successful and charged update counts, worker count, GPU activity,
VRAM and CPU/RAM. A heartbeat explicitly says when there is **no new progress**. Status continues
during a blocked Drive request; it does not mean Drive is healthy. Worker phases distinguish
initialization, loading, calibration, training, evaluation, checkpointing and waiting for Drive.
Timing covers data loading, evaluation, ledger writes/readback, checkpoints and recovery uploads.
These are wall-clock timings; overlapping upload time is not additive and no GPU synchronizations
or profiling trials are inserted. GPU utilization measures activity; VRAM measures capacity.
Local live files are `/content/hippo_adni_bundle/parallel_status/<session>/status.json`
and `timing.jsonl`; worker statuses are in the parent folder. Per-run measurements are archived,
and session telemetry is copied to Drive at a clean stop. A hard VM loss can lose local telemetry,
but never relaxes the original durable scientific-accounting requirements.
Existing completed archives are never rewritten. No extra scientific runs or tuning trials are added.

**To stop:** interrupt once and wait for every worker's recovery upload. A worker failure stops new
launches and asks the others to save. Hard VM loss still requires an audit if charged updates are
newer than durable checkpoints; accounting is never reset to hide lost work.

`SESSION_HOURS` counts from starting this cell. Set it below your **remaining** runtime allowance,
including time already spent in the previous notebook. The default is 8 more hours for this handover;
on a newly allocated Pro+ runtime you may use 22. Colab can terminate earlier. Existing STOP files are respected.


In [ ]:
MAX_NEW_RUNS = 504  # Upper bound on remaining tasks launched this session, not extra repeats.
MAX_WORKERS = 0    # Immediately target CPU-supported maximum; resource checks gate occupancy.
SESSION_HOURS = 8.0  # Remaining time for this session; fresh runtime may use 22.0.
campaign('run', '--max-runs', MAX_NEW_RUNS, '--hours', SESSION_HOURS, '--workers', MAX_WORKERS)


## 6. Results so far
Each gain is paired with the same model/fold/seed baseline. A final mean is shown only after all
nine cells of that model/condition are present: average the three seeds within each fold, then
average the three fold means. The displayed SD is across fold means, not nine independent subjects.
Architecture remains confounded with baseline loss/optimizer; this is validation, not an independent test set.


In [ ]:
campaign('report')
result = json.loads((RESULTS/'comparison.json').read_text())
import pandas as pd
print(f"Complete: {result['completed']}/{result['total']}")
display(pd.DataFrame(result['aggregate']))
display(pd.DataFrame(result['runs']))


## 7. Curves and GPU use
Per-run `metrics.csv`, `completion.json`, `calibration.json`, full diagnostics, configuration and
integer FN/FP live under `results_noaug_v1/runs/`. Inspect training and validation together. ADNI has one foreground class; A/P swaps do not apply.
Use the runtime's GPU utilization display or `nvidia-smi`; high utilization alone does not establish
a faster or scientifically equivalent experiment. Do not raise batch size or change precision.

Scheduler provenance, preservation inventory and session timing are under `results_noaug_v1/parallel/`.


In [ ]:
import matplotlib.pyplot as plt
finished = result['runs']
if finished:
    chosen = finished[-1]['id']
    curve = pd.read_csv(RESULTS/'runs'/chosen/'metrics.csv')
    plt.plot(curve['compute_fraction'], 100*curve['macro_hard_dice'], marker='.')
    plt.xlabel('Fraction of optimization FLOP budget')
    plt.ylabel('Validation Dice (%)')
    plt.title(chosen)
    plt.grid(alpha=.25)
    plt.show()
subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total', '--format=csv'], check=True)


## Resume later
Open this same notebook, use the same GPU type, and rerun setup/preflight/run cells. Keep the same
results folder and uploaded archives. No laptop collector or cluster scheduler is involved.
Do not run two copies against the same output directory. A changed GPU/software environment is
rejected instead of mixing its results into the nine-cell comparisons.

Google Drive's quota API is documented [here](https://developers.google.com/workspace/drive/api/reference/rest/v3/about).
Colab resources and background execution remain subject to the [Colab FAQ](https://research.google.com/colaboratory/faq.html).
Disconnect and delete the GPU runtime when finished; the verified Drive results remain.


## 8. Export validation evidence without extra runs

After a normal run cell finishes or stops cleanly, run this section and send the downloaded
`adni_colab_validation_v5.zip` back in chat. It contains only existing local controller logs,
worker progress, memory/timing records, environment binding and storage placement details.
It launches no jobs, calls no model, reads no Drive files and does not alter results or accounting.
This validates observed scheduling/storage behavior; it does not certify new GPU acceleration.
If training failed, you can still run this section to collect the local diagnostic evidence.


In [ ]:
VALIDATION_ZIP = Path('/content/adni_colab_validation_v5.zip')
subprocess.run([str(PYTHON), str(CONTROL/'validation.py'), '--bundle', str(BUNDLE),
                '--control', str(CONTROL), '--output', str(VALIDATION_ZIP),
                '--log', '/content/hippo_adni_run.log'], check=True)
from google.colab import files
files.download(str(VALIDATION_ZIP))
